In [1]:
import pandas as pd

# Read CSV
df = pd.read_csv("fcpeuro_bmw-spark-plug-kit_url_output.csv")

# Keep Year as its own column
# Combine only Fitment_Application + Fitment_Additional
df["Combined_Fitment"] = (
    df["Fitment_Application"].astype(str) + " | " +
    df["Fitment_Additional"].astype(str)
)

# Group and aggregate URLs into lists
grouped = df.groupby(["Year", "Combined_Fitment"]).agg(
    URL_List=("URL", list)
).reset_index()

# Compute URL count per row
grouped["URL_Count"] = grouped["URL_List"].apply(len)

# Expand the URL lists into separate columns
urls_expanded = pd.DataFrame(grouped["URL_List"].to_list())
urls_expanded.columns = [f"URL_{i+1}" for i in range(urls_expanded.shape[1])]

# Combine with Year, Combined_Fitment, and URL_Count
result = pd.concat([grouped[["Year", "Combined_Fitment", "URL_Count"]], urls_expanded], axis=1)

# Export to CSV
result.to_csv("output/fcpeuro_bmw-spark-plug-kit_url_grouped_wide.csv", index=False)

print(result)


         Year                                   Combined_Fitment  URL_Count  \
0      2000.0  2000 BMW 323Ci Spark Plug Connector | Automati...          1   
1      2000.0  2000 BMW 323Ci Spark Plug Connector | Automati...          1   
2      2000.0  2000 BMW 323Ci Spark Plug Connector | Manual R...          1   
3      2000.0  2000 BMW 323Ci Spark Plug Connector | Manual R...          1   
4      2000.0  2000 BMW 323Ci Spark Plug Set | Automatic RWD ...          2   
...       ...                                                ...        ...   
15300  2026.0  2026 BMW M4 Competition xDrive Spark Plug | Au...          1   
15301  2026.0  2026 BMW M4 Competition xDrive Spark Plug | Au...          1   
15302  2026.0  2026 BMW M4 Spark Plug | Manual RWD Coupe 3.0L...          1   
15303  2026.0  2026 BMW Z4 M40i Spark Plug | Automatic RWD Co...          1   
15304  2026.0  2026 BMW Z4 sDrive30i Spark Plug | Automatic R...          1   

                                                   

In [1]:
import pandas as pd

# Read input CSV
df = pd.read_csv("output/fcpeuro_audi-spark-plug-kit_output.csv")

# Group by two columns and aggregate URLs into lists
grouped = df.groupby(["Fitment_Application", "Fitment_Additional"]).agg(
    URL_List=("URL", list)
).reset_index()

# Explode the URL list so each URL has its own row
result = grouped.explode("URL_List").rename(columns={"URL_List": "URL"})

# Add a URL count per group (optional)
result["URL_Count"] = result.groupby(
    ["Fitment_Application", "Fitment_Additional"]
)["URL"].transform("count")

# Reorder columns
result = result[["Fitment_Application", "Fitment_Additional", "URL_Count", "URL"]]

# Print result
print(result)

# Export to CSV
result.to_csv("output/fcpeuro_audi-spark-plug-kit_groupby_pdp.csv", index=False)


                           Fitment_Application  \
0        2004 Audi A8 Quattro L Spark Plug Set   
0        2004 Audi A8 Quattro L Spark Plug Set   
0        2004 Audi A8 Quattro L Spark Plug Set   
1     2004 Audi Allroad Quattro Spark Plug Set   
1     2004 Audi Allroad Quattro Spark Plug Set   
...                                        ...   
1347  2026 Audi Q8 Premium Plus Spark Plug Set   
1348       2026 Audi Q8 Premium Spark Plug Set   
1348       2026 Audi Q8 Premium Spark Plug Set   
1349      2026 Audi Q8 Prestige Spark Plug Set   
1349      2026 Audi Q8 Prestige Spark Plug Set   

                                     Fitment_Additional  URL_Count  \
0                   Automatic AWD Sedan 4.2L V8 BFM Gas          3   
0                   Automatic AWD Sedan 4.2L V8 BFM Gas          3   
0                   Automatic AWD Sedan 4.2L V8 BFM Gas          3   
1                   Automatic AWD Wagon 4.2L V8 BAS Gas          3   
1                   Automatic AWD Wagon 4.2L V8 B